<a href="https://colab.research.google.com/github/sebastianatanasovici-wq/-notebook-/blob/main/pregunta_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from __future__ import annotations

from functools import wraps
from time import perf_counter


In [2]:
%pip install amplpy
try:
    from amplpy import ampl_notebook
except ImportError as exc:
    raise ImportError(
        "This notebook requires amplpy. Install it with `%pip install amplpy` before running."
    ) from exc


def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed
    return wrapper


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 11.6 MB/s eta 0:00:00


**1)La Química fabrica tres productos químicos**

**Proceso 1:**

Costo: 40

Produce: 3A, 1B, 1C

**Proceso 2:**

Costo: 10

Produce: 1A, 1B, 0C

**Demandas mínimas:**

A ≥ 40
B ≥ 15
C ≥ 5

x1:proceso 1

x2:proceso 2

Restricciones activas:

Producción de C: x1 ≥ 5
Producción de A: 3x1 + x2 ≥ 40

No activa:

Producción de B

Interpretación:

La producción de C obliga a usar el proceso 1

El proceso 2 se usa al máximo porque es más barato

Sensibilidad:

Si aumenta la demanda de C → aumenta el costo directamente

Si baja el costo del proceso 1 → se usaría más

Si sube el costo del proceso 2 → cambia la mezcla óptima

In [4]:
def create_ampl_instance(solver: str = 'highs'):
    return ampl_notebook(modules=[solver], license_uuid='default')


def extract_solution(ampl):
    x1 = ampl.get_variable("x1").value()
    x2 = ampl.get_variable("x2").value()
    cost = ampl.get_objective("Costo_Total").value()

    return {
        "Proceso_1_horas": round(x1, 2),
        "Proceso_2_horas": round(x2, 2),
        "Costo_Total": round(cost, 2),
    }


@timed
def solve_quimica_problem():
    ampl = create_ampl_instance()

    ampl.eval(r'''
        # VARIABLES
        var x1 >= 0;
        var x2 >= 0;

        # FUNCION OBJETIVO
        minimize Costo_Total:
            40*x1 + 10*x2;

        # RESTRICCIONES
        r1:
            3*x1 + x2 >= 40;

        r2:
            x1 + x2 >= 15;

        r3:
            x1 >= 5;
    ''')
    ampl.option["solver"] = "highs"
    ampl.solve()

    return extract_solution(ampl)


resultado, tiempo = solve_quimica_problem()

print("=== RESULTADOS EJERCICIO 1 ===")
print(f"Solución -> {resultado}")
print(f"Tiempo   -> {tiempo:.6f} segundos")

Using default Community Edition License for Colab. Get yours at: https://ampl.com/ce
Licensed to AMPL Community Edition License for the AMPL Model Colaboratory (https://ampl.com/colab).
HiGHS 1.11.0: HiGHS 1.11.0: optimal solution; objective 450
0 simplex iterations
0 barrier iterations
=== RESULTADOS EJERCICIO 1 ===
Solución -> {'Proceso_1_horas': 5.0, 'Proceso_2_horas': 25.0, 'Costo_Total': 450.0}
Tiempo   -> 7.186334 segundos
